STEP 1: LOAD And EXPLORE

In [1]:
#Cell 1: load and target balance
import numpy as np
import pandas as pd

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
print(train.shape, test.shape)
print(train["Transported"].value_counts(normalize=True).round(3))
#Cell 2: missing values, train vs testprint(train["Transported"].value_counts(normalize=True).round(3))
train.head(10)

(8693, 14) (4277, 13)
Transported
True     0.504
False    0.496
Name: proportion, dtype: float64


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True


In [2]:
#Cell 2: missing values, train vs test

# AUDIT: Compare missing value proportions side-by-side between train and test
# --- Method Breakdown ---
# 1. isna()  : Detects missing cells (returns True if missing, False if filled)
# 2. mean()  : Computes missing percentage (Python counts True=1, False=0; 
#              so the mean gives the exact fraction of missing rows, e.g., 0.025 = 2.5%)
# 3. concat  : Glues the train and test series together side-by-side (axis=1)
# 4. keys    : Assigns column header names ("train" and "test") to the combined table
# 5. round(3): Keeps decimals clean to 3 places
miss = pd.concat([train.isna().mean(), test.isna().mean()],
                 axis=1, keys=["train", "test"]).round(3) 
print(miss.sort_values("train", ascending=False))
test.head(10)

              train   test
CryoSleep     0.025  0.022
ShoppingMall  0.024  0.023
Cabin         0.023  0.023
Name          0.023  0.022
VIP           0.023  0.022
HomePlanet    0.023  0.020
VRDeck        0.022  0.019
FoodCourt     0.021  0.025
Destination   0.021  0.022
Age           0.021  0.021
Spa           0.021  0.024
RoomService   0.021  0.019
PassengerId   0.000  0.000
Transported   0.000    NaN


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez
5,0027_01,Earth,False,F/7/P,TRAPPIST-1e,31.0,False,0.0,1615.0,263.0,113.0,60.0,Karlen Ricks
6,0029_01,Europa,True,B/2/P,55 Cancri e,21.0,False,0.0,NaN,0.0,0.0,0.0,Aldah Ainserfle
7,0032_01,Europa,True,D/0/S,TRAPPIST-1e,20.0,False,0.0,0.0,0.0,0.0,0.0,Acrabi Pringry
8,0032_02,Europa,True,D/0/S,55 Cancri e,23.0,False,0.0,0.0,0.0,0.0,0.0,Dhena Pringry
9,0033_01,Earth,False,F/7/S,55 Cancri e,24.0,False,0.0,639.0,0.0,0.0,0.0,Eliana Delazarson


In [3]:
#Cell 3: unpack the hidden structure (exploration only)

def parse(df):
    df = df.copy()
    df[["Group", "GroupPos"]] = df["PassengerId"].str.split("_", expand=True).astype(int)
    df[["Deck", "CabinNum", "Side"]] = df["Cabin"].str.split("/", expand=True)
    return df

tr = parse(train)
print(tr[["PassengerId", "Group", "GroupPos", "Cabin", "Deck", "CabinNum", "Side"]].head())
print("Unique groups:", tr["Group"].nunique(), "of", len(tr), "passengers")

  PassengerId  Group  GroupPos  Cabin Deck CabinNum Side
0     0001_01      1         1  B/0/P    B        0    P
1     0002_01      2         1  F/0/S    F        0    S
2     0003_01      3         1  A/0/S    A        0    S
3     0003_02      3         2  A/0/S    A        0    S
4     0004_01      4         1  F/1/S    F        1    S
Unique groups: 6217 of 8693 passengers


In [4]:
#Cell 4: which categories relate to the target?
for col in ["HomePlanet", "CryoSleep", "Destination", "Deck", "Side", "VIP"]:
    print(tr.groupby(col, dropna=False)["Transported"].agg(["mean", "count"]).round(3), "\n")

             mean  count
HomePlanet              
Earth       0.424   4602
Europa      0.659   2131
Mars        0.523   1759
NaN         0.512    201 

            mean  count
CryoSleep              
False      0.329   5439
True       0.818   3037
NaN        0.488    217 

                mean  count
Destination                
55 Cancri e    0.610   1800
PSO J318.5-22  0.504    796
TRAPPIST-1e    0.471   5915
NaN            0.505    182 

       mean  count
Deck              
A     0.496    256
B     0.734    779
C     0.680    747
D     0.433    478
E     0.357    876
F     0.440   2794
G     0.516   2559
T     0.200      5
NaN   0.503    199 

       mean  count
Side              
P     0.451   4206
S     0.555   4288
NaN   0.503    199 

        mean  count
VIP                
False  0.506   8291
True   0.382    199
NaN    0.512    203 



In [5]:
#Cell 5: spending and age
spend = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
tr["TotalSpend"] = tr[spend].sum(axis=1)

# Do CryoSleep passengers spend nothing? (they were in suspended animation)
print("Share of CryoSleep=True passengers with zero spend:",
      (tr.loc[tr["CryoSleep"] == True, "TotalSpend"] == 0).mean().round(3))

print(tr.groupby(pd.cut(tr["Age"], [0, 12, 18, 30, 50, 90]), observed=True)["Transported"]
        .agg(["mean", "count"]).round(3))
print(tr.groupby(tr["TotalSpend"] == 0)["Transported"].mean().round(3))

Share of CryoSleep=True passengers with zero spend: 1.0
           mean  count
Age                   
(0, 12]   0.669    628
(12, 18]  0.537   1059
(18, 30]  0.468   3238
(30, 50]  0.479   2674
(50, 90]  0.484    737
TotalSpend
False    0.299
True     0.786
Name: Transported, dtype: float64


In [6]:
#Cell 6: groups and the leakage question
grp = tr.groupby("Group")
print("Group size distribution:\n", grp.size().value_counts().sort_index())

# Do group members tend to share the outcome?
multi = grp.filter(lambda g: len(g) > 1)
agree = multi.groupby("Group")["Transported"].agg(lambda s: s.nunique() == 1).mean()
print("Share of multi-person groups where everyone has the same outcome:", round(agree, 3))

# Do groups in train also appear in test?
te = parse(test)
print("Groups appearing in BOTH train and test:", len(set(tr["Group"]) & set(te["Group"])))

Group size distribution:
 1    4805
2     841
3     340
4     103
5      53
6      29
7      33
8      13
Name: count, dtype: int64
Share of multi-person groups where everyone has the same outcome: 0.436
Groups appearing in BOTH train and test: 0


In [7]:
# CELL 7 (optional): what agreement would we see if outcomes were independent of group?
# Why: 43.6% means nothing without a chance baseline. Shuffling the labels breaks any
# real link to groups, so the shuffled agreement rate IS the chance level.
rng = np.random.default_rng(42)
y_all = tr["Transported"].astype(int).values
multi_mask = tr.groupby("Group")["Group"].transform("size") > 1
g_multi = tr.loc[multi_mask, "Group"]

def agree_share(labels):
    s = pd.Series(labels, index=tr.index)[multi_mask]
    return s.groupby(g_multi).nunique().eq(1).mean()

observed = agree_share(y_all)
chance = [agree_share(rng.permutation(y_all)) for _ in range(200)]
print("observed:", round(observed, 3), "| chance mean:", round(np.mean(chance), 3),
      "| chance 95% range:", np.percentile(chance, [2.5, 97.5]).round(3))

observed: 0.436 | chance mean: 0.37 | chance 95% range: [0.348 0.396]


Step 2: Grouped CV, baselines, first submission

In [8]:
#Cell 1: setup, target, groups, and logging helper
import numpy as np, pandas as pd, os
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SEED = 42
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

y = train["Transported"].astype(int)
# Group id = first part of PassengerId ("0003_02" -> 3). Used ONLY to keep groups together in CV.
# It is not a model feature yet.
groups = train["PassengerId"].str.split("_").str[0].astype(int)

LOG = "../reports/experiment_log.csv"
def log(step, change, cv_mean, cv_std, lb="", kept="", why=""):
    """Append one row to the experiment log. Why: the write-up's 'what improved it' table
    is built from this file, so every experiment is recorded at the time it happens."""
    row = pd.DataFrame([dict(step=step, change=change, cv_mean=round(cv_mean, 4),
                             cv_std=round(cv_std, 4), public_lb=lb, kept=kept, why=why)])
    row.to_csv(LOG, mode="a", header=not os.path.exists(LOG), index=False)


In [9]:
#Cell 2: the CV function
# DECISION: StratifiedGroupKFold, 5 folds, shuffled, repeated with 3 different seeds.
# Why each part:
#  - Group: whole groups stay in one fold, matching the real train/test split (0 shared groups).
#  - Stratified: keeps the ~50/50 class balance in each fold, so fold scores aren't skewed.
#  - Shuffle: PassengerId order may correlate with cabin or planet; without shuffling, folds
#    would be contiguous blocks of that order and wouldn't be representative.
#  - 3 repeats (15 scores): one 5-fold run has noise of about ±0.01, which hides small
#    improvements. Repeats shrink that noise. The same seeds are used for every model, so
#    comparisons are paired and fair.
def cv_accuracy(model, X, grouped=True, n_repeats=3):
    scores = []
    for r in range(n_repeats):
        if grouped:
            splits = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r)
                          .split(X, y, groups))
        else:
            splits = list(StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(X, y))
        scores.extend(cross_val_score(model, X, y, cv=splits, scoring="accuracy", n_jobs=-1))
    return np.mean(scores), np.std(scores)

In [10]:
#Cell 3: the deliberately raw baseline
# DECISION: the baseline uses ONLY the raw columns, with no engineering at all.
# Why: it is the "before" number. Every later improvement is measured against it, so the
# write-up can say exactly what each change added. We drop Name (an identifier), PassengerId
# and Cabin (packed strings the model can't use as-is; we unpack them in Step 4).
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def prep_raw(df):
    df = df[["HomePlanet", "CryoSleep", "Destination", "VIP", "Age"] + SPEND].copy()
    # CryoSleep/VIP hold True/False/NaN objects. Mapping to strings keeps NaN as a proper
    # missing value so the imputer can handle it, and lets one-hot treat them as categories.
    for c in ["CryoSleep", "VIP"]:
        df[c] = df[c].map({True: "True", False: "False"})
    return df

X_raw = prep_raw(train)

def make_preprocessor(scale):
    # Numeric: median is robust to the heavy skew in spending. Scaling only for logistic
    # regression, because its penalty depends on feature scale and trees don't care.
    num = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num), make_column_selector(dtype_include="number")),
        # Categorical: most frequent value; handle_unknown="ignore" avoids crashes on rare
        # levels that appear in only one fold or only in the test set.
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
         make_column_selector(dtype_exclude="number")),
    ])

models = {
    # Floor: always predicts the majority class. Anything real must beat this clearly.
    "Dummy (majority)": Pipeline([("prep", make_preprocessor(False)),
                                  ("m", DummyClassifier(strategy="most_frequent"))]),
    # Simple linear baseline.
    "Logistic (raw)": Pipeline([("prep", make_preprocessor(True)),
                                ("m", LogisticRegression(max_iter=2000))]),
    # Nonlinear baseline. min_samples_leaf=3 limits overfitting; n_jobs=1 because
    # cross_val_score already parallelises across folds.
    "Random Forest (raw)": Pipeline([("prep", make_preprocessor(False)),
                                     ("m", RandomForestClassifier(n_estimators=300,
                                                                  min_samples_leaf=3,
                                                                  random_state=SEED, n_jobs=1))]),
}
for name, m in models.items():
    mean, std = cv_accuracy(m, X_raw)
    print(f"{name:22s} grouped CV accuracy {mean:.4f} ± {std:.4f}")
    log("2 baseline", name, mean, std)

Dummy (majority)       grouped CV accuracy 0.5036 ± 0.0001
Logistic (raw)         grouped CV accuracy 0.7863 ± 0.0095
Random Forest (raw)    grouped CV accuracy 0.7941 ± 0.0095


In [11]:
#Cell 4: how much does the CV scheme matter?
# Why: this quantifies the leakage concern. If random CV scores higher than grouped CV for
# the same model, the gap is leakage-inflated optimism. For the write-up this is evidence for
# why grouped CV was needed, not just a claim. Expect a SMALL gap now, because raw features
# contain nothing about groups. The gap should widen once we add cabin and group features.
for name in ["Logistic (raw)", "Random Forest (raw)"]:
    g = cv_accuracy(models[name], X_raw, grouped=True)
    r = cv_accuracy(models[name], X_raw, grouped=False)
    print(f"{name:22s} grouped {g[0]:.4f} | random {r[0]:.4f} | gap {r[0]-g[0]:+.4f}")

Logistic (raw)         grouped 0.7863 | random 0.7861 | gap -0.0002
Random Forest (raw)    grouped 0.7941 | random 0.7909 | gap -0.0032


In [12]:
#Cell 5: write the first submission
# DECISION: submit the baseline with the better CV score, chosen by CV and not by trying
# both on the leaderboard. Why: the public leaderboard is only a fraction of the test rows
# and limited in daily submissions. Choosing between models by repeatedly checking it would
# overfit to it. Here the leaderboard is used only to verify that our CV estimate is realistic.
best_name = "Random Forest (raw)"   # <- change to whichever had the higher CV in Cell 3
final = models[best_name].fit(X_raw, y)
pred = final.predict(prep_raw(test)).astype(bool)

sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Transported": pred})
os.makedirs("../submissions", exist_ok=True)
sub.to_csv("../submissions/sub01_baseline.csv", index=False)
print(sub.shape, sub["Transported"].mean().round(3))
print(sub.head())

(4277, 2) 0.522
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01        False


In [13]:
# Why: the write-up's results table needs the leaderboard score next to the CV score it came from.
lg = pd.read_csv(LOG)
lg.loc[lg["change"] == "Random Forest (raw)", "public_lb"] = 0.7884
lg.to_csv(LOG, index=False)

Step 4: Feature engineering, one idea at a time

In [14]:
#Cell 1: verify the groupfill assumption
# Why: groupfill is only valid if group members really share a home planet. Check, don't assume.
full = pd.concat([train.drop(columns="Transported"), test], ignore_index=True)
g = full["PassengerId"].str.split("_").str[0].astype(int)
hp = full.groupby(g)["HomePlanet"].nunique()
print("Groups with more than one distinct HomePlanet:", (hp > 1).sum(), "of", len(hp))

Groups with more than one distinct HomePlanet: 0 of 9280


In [25]:
#Cell 2: the feature builder
def build_features(train, test, blocks):
    full = pd.concat([train.drop(columns="Transported"), test], ignore_index=True)
    grp = full["PassengerId"].str.split("_").str[0].astype(int)
    df = full[["HomePlanet", "CryoSleep", "Destination", "VIP", "Age"] + SPEND].copy()
    for c in ["CryoSleep", "VIP"]:
        df[c] = full[c].map({True: "True", False: "False"})

    if "groupfill" in blocks:
        # "first" returns the first non-missing value in each group.
        df["HomePlanet"] = df["HomePlanet"].fillna(df.groupby(grp)["HomePlanet"].transform("first"))

    if "rules" in blocks:
        # Order matters: repair CryoSleep first (anyone who spent money was not in cryosleep),
        # then use it to zero the missing spend of cryosleepers.
        df.loc[df["CryoSleep"].isna() & (df[SPEND].sum(axis=1) > 0), "CryoSleep"] = "False"
        cryo = df["CryoSleep"] == "True"
        for c in SPEND:
            df.loc[cryo & df[c].isna(), c] = 0.0
    
        # Put this right AFTER the existing `rules` block.
    if "rules2" in blocks:
        # Why: among passengers with all five spends observed and zero, 85.1% were in
        # cryosleep, yet the mode imputer fills a missing CryoSleep with False for them.
        # Uses only other features, never Transported, so there is no label leakage.
        obs_zero = df[SPEND].notna().all(axis=1) & (df[SPEND].sum(axis=1) == 0)
        df.loc[df["CryoSleep"].isna() & obs_zero, "CryoSleep"] = "True"

        # Split the old 'spend' block into three single-idea blocks to find which part hurt Logistic.
    if {"nospend", "totspend"} & set(blocks):
        total = df[SPEND].sum(axis=1)          # computed BEFORE any log transform
        if "totspend" in blocks:
            df["TotalSpend"] = total           # raw total: tests the heavy-tail hypothesis
        if "nospend" in blocks:
            df["NoSpend"] = (total == 0).astype(int)
    if "logspend" in blocks:
        for c in SPEND:                        # log of each column: tests the reshaping hypothesis
            df[c] = np.log1p(df[c])

    if "age" in blocks:
        df["IsChild"] = (df["Age"] <= 12).astype(int)

    if "cabin" in blocks:
        cab = full["Cabin"].str.split("/", expand=True)
        df["Deck"], df["CabinNum"], df["Side"] = cab[0], pd.to_numeric(cab[1]), cab[2]

    # Put this right AFTER the existing `cabin` block (it needs CabinNum).
    if "cabinbin" in blocks:
        # Why: the transported rate is a wave across cabin numbers, not a slope. One-hot
        # bins of fixed width 100 (chosen up front, not tuned to the labels) let Logistic
        # fit a separate level per range. CabinNum itself stays as a numeric feature.
        if "CabinNum" not in df:
            cab = full["Cabin"].str.split("/", expand=True)
            df["Deck"], df["CabinNum"], df["Side"] = cab[0], pd.to_numeric(cab[1]), cab[2]
        b = df["CabinNum"] // 100
        df["CabinBin"] = b.map(lambda v: f"bin{int(v)}" if pd.notna(v) else np.nan)

    if "group" in blocks:
        df["GroupSize"] = grp.map(grp.value_counts())
        df["IsAlone"] = (df["GroupSize"] == 1).astype(int)

    if "family" in blocks:
        sur = full["Name"].str.split().str[-1]
        df["FamilySize"] = sur.map(sur.value_counts()).fillna(1)   # missing name -> assume alone

    n = len(train)
    return df.iloc[:n].reset_index(drop=True), df.iloc[n:].reset_index(drop=True)

def make_models():
    return {
        "Logistic": Pipeline([("prep", make_preprocessor(True)),
                              ("m", LogisticRegression(max_iter=3000))]),
        "RF": Pipeline([("prep", make_preprocessor(False)),
                        ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                     random_state=SEED, n_jobs=1))]),
    }

def run(blocks, grouped=True):
    Xtr, _ = build_features(train, test, blocks)
    return {name: cv_accuracy(m, Xtr, grouped=grouped) for name, m in make_models().items()}

In [16]:
#Cell 3: sanity check, then cumulative ablation
# Why: build_features with NO blocks must reproduce the Step 2 baseline (0.7863 / 0.7941).
# If it doesn't, the new code differs from the old in some way and every later number is suspect.
order = ["rules", "spend", "age", "cabin", "group", "groupfill", "family"]
blocks, prev = (), run(())
print(f"{'raw (check)':18s}", {k: round(v[0], 4) for k, v in prev.items()})
for name, (m, s) in prev.items():
    log("4 features", f"raw rebuilt [{name}]", m, s)

for b in order:
    blocks += (b,)
    cur = run(blocks)
    print(f"{'+ ' + b:18s}", {k: f"{v[0]:.4f} ({v[0]-prev[k][0]:+.4f})" for k, v in cur.items()})
    for name, (m, s) in cur.items():
        log("4 features", f"+ {b} (cumulative) [{name}]", m, s)
    prev = cur

raw (check)        {'Logistic': np.float64(0.7863), 'RF': np.float64(0.7941)}
+ rules            {'Logistic': '0.7863 (+0.0000)', 'RF': '0.7941 (+0.0000)'}
+ spend            {'Logistic': '0.7863 (+0.0000)', 'RF': '0.7941 (+0.0000)'}
+ age              {'Logistic': '0.7842 (-0.0021)', 'RF': '0.7945 (+0.0004)'}
+ cabin            {'Logistic': '0.7929 (+0.0087)', 'RF': '0.8092 (+0.0147)'}
+ group            {'Logistic': '0.7937 (+0.0008)', 'RF': '0.8069 (-0.0023)'}
+ groupfill        {'Logistic': '0.7935 (-0.0002)', 'RF': '0.8066 (-0.0003)'}
+ family           {'Logistic': '0.7934 (-0.0000)', 'RF': '0.8073 (+0.0007)'}


In [17]:
#Cell 4: leave-one-block-out, the fairer check
# Why: from the full set, remove ONE block at a time. This shows each block's contribution
# given all the others, which doesn't depend on the order in which blocks were added.
FULL = tuple(order)
full_res = run(FULL)
print("FULL:", {k: round(v[0], 4) for k, v in full_res.items()})
for b in order:
    r = run(tuple(x for x in FULL if x != b))
    print(f"without {b:10s}", {k: f"{v[0]:.4f} ({v[0]-full_res[k][0]:+.4f})" for k, v in r.items()})

FULL: {'Logistic': np.float64(0.7934), 'RF': np.float64(0.8073)}
without rules      {'Logistic': '0.7934 (+0.0000)', 'RF': '0.8073 (+0.0000)'}
without spend      {'Logistic': '0.7934 (+0.0000)', 'RF': '0.8073 (+0.0000)'}
without age        {'Logistic': '0.7904 (-0.0030)', 'RF': '0.8064 (-0.0009)'}
without cabin      {'Logistic': '0.7874 (-0.0060)', 'RF': '0.7891 (-0.0181)'}
without group      {'Logistic': '0.7930 (-0.0005)', 'RF': '0.8087 (+0.0014)'}
without groupfill  {'Logistic': '0.7936 (+0.0001)', 'RF': '0.8077 (+0.0004)'}
without family     {'Logistic': '0.7935 (+0.0000)', 'RF': '0.8066 (-0.0007)'}


In [18]:
#Cell 5: does the leakage gap appear now?
# Why: tests the hypothesis from the baseline. With cabin and group features present,
# a random split should score HIGHER than a grouped one, because the model can recognise
# a passenger's companions. If it does, that gap is evidence that grouped CV was needed.
g = run(FULL, grouped=True)
r = run(FULL, grouped=False)
for k in g:
    print(f"{k:9s} grouped {g[k][0]:.4f} | random {r[k][0]:.4f} | gap {r[k][0]-g[k][0]:+.4f}")

Logistic  grouped 0.7934 | random 0.7927 | gap -0.0007
RF        grouped 0.8073 | random 0.8067 | gap -0.0005


Step 4b: Confirm, diagnose, then submit

In [19]:
#Cell 1: verify the rules explanation
# Why: the no-op explanation is a claim; this checks it instead of asserting it.
print(train[SPEND].median())          # expect 0.0 for all five

# Bonus: the mode imputer assumes CryoSleep=False when unknown. Is that right for
# passengers whose spending is observed and zero?
zero = train[SPEND].notna().all(axis=1) & (train[SPEND].sum(axis=1) == 0)
print(train.loc[zero, "CryoSleep"].value_counts(normalize=True).round(3))

RoomService     0.0
FoodCourt       0.0
ShoppingMall    0.0
Spa             0.0
VRDeck          0.0
dtype: float64
CryoSleep
True     0.851
False    0.149
Name: proportion, dtype: float64


In [20]:
#Cell 2: split spend into single ideas, on top of SELECTED
SELECTED = ("rules", "groupfill", "cabin")
base = run(SELECTED)
print("SELECTED:", {k: round(v[0], 4) for k, v in base.items()})
for k, (m, s) in base.items():
    log("4 features", f"SELECTED = rules+groupfill+cabin [{k}]", m, s, kept="yes",
        why="only block passing the 0.002 rule; others neutral or harmful")

for extra in ["nospend", "totspend", "logspend"]:
    r = run(SELECTED + (extra,))
    print(f"+ {extra:9s}", {k: f"{v[0]:.4f} ({v[0]-base[k][0]:+.4f})" for k, v in r.items()})
    for k, (m, s) in r.items():
        log("4b spend diagnostic", f"SELECTED + {extra} [{k}]", m, s)

SELECTED: {'Logistic': np.float64(0.7904), 'RF': np.float64(0.8088)}
+ nospend   {'Logistic': '0.7903 (-0.0002)', 'RF': '0.8088 (+0.0000)'}
+ totspend  {'Logistic': '0.7908 (+0.0003)', 'RF': '0.8095 (+0.0007)'}
+ logspend  {'Logistic': '0.7779 (-0.0125)', 'RF': '0.8087 (-0.0001)'}


In [21]:
#Cell 3: is CabinNum non-linear? (exploration only)
# Why: tests the hypothesis from finding 1. Fixed-width bins (100) are chosen up front and
# not tuned to the labels, so we don't leak the target into the feature design.
Xc, _ = build_features(train, test, ("cabin",))
t = Xc.assign(y=y.values)
t["CabinBin"] = t["CabinNum"] // 100
print(t.groupby("CabinBin")["y"].agg(["mean", "count"]).round(3))

           mean  count
CabinBin              
0.0       0.530   1331
1.0       0.538   1164
2.0       0.548   1065
3.0       0.434    564
4.0       0.362    458
5.0       0.433    466
6.0       0.503    322
7.0       0.542    325
8.0       0.584    329
9.0       0.666    350
10.0      0.635    334
11.0      0.506    308
12.0      0.437    309
13.0      0.417    307
14.0      0.407    322
15.0      0.435    154
16.0      0.311    148
17.0      0.481    158
18.0      0.438     80


In [23]:
# Why: caps the possible gain of rules2 before we run it, and checks whether cabin bins
# are really deck effects in disguise.
obs_zero = train[SPEND].notna().all(axis=1) & (train[SPEND].sum(axis=1) == 0)
print("CryoSleep missing:", train["CryoSleep"].isna().sum(),
      "| fillable by rules2:", (train["CryoSleep"].isna() & obs_zero).sum())

Xc, _ = build_features(train, test, ("cabin",))
print(pd.crosstab(Xc["CabinNum"] // 100, Xc["Deck"]))

CryoSleep missing: 217 | fillable by rules2: 87
Deck        A    B    C    D    E    F    G  T
CabinNum                                      
0.0       243  233  207  168  151  159  165  5
1.0        13  240  248  173  149  157  184  0
2.0         0  242  223  137  136  156  171  0
3.0         0   64   69    0  144  130  157  0
4.0         0    0    0    0  152  146  160  0
5.0         0    0    0    0  133  154  179  0
6.0         0    0    0    0   11  129  182  0
7.0         0    0    0    0    0  163  162  0
8.0         0    0    0    0    0  151  178  0
9.0         0    0    0    0    0  170  180  0
10.0        0    0    0    0    0  145  189  0
11.0        0    0    0    0    0  168  140  0
12.0        0    0    0    0    0  151  158  0
13.0        0    0    0    0    0  140  167  0
14.0        0    0    0    0    0  142  180  0
15.0        0    0    0    0    0  147    7  0
16.0        0    0    0    0    0  148    0  0
17.0        0    0    0    0    0  158    0  0
18.0        

In [26]:
SELECTED = ("rules", "groupfill", "cabin")
base = run(SELECTED)
for extra in ["rules2", "cabinbin"]:
    r = run(SELECTED + (extra,))
    print(f"+ {extra:9s}", {k: f"{v[0]:.4f} ({v[0]-base[k][0]:+.4f})" for k, v in r.items()})
    for k, (m, s) in r.items():
        log("4c", f"SELECTED + {extra} [{k}]", m, s)

+ rules2    {'Logistic': '0.7906 (+0.0002)', 'RF': '0.8081 (-0.0007)'}
+ cabinbin  {'Logistic': '0.7950 (+0.0046)', 'RF': '0.8055 (-0.0033)'}


In [ ]:
# DECISION: RF is the submitted model (higher CV). FINAL = SELECTED, plus a block ONLY if it
# passed the 0.002 rule FOR RF. A block that helps only Logistic doesn't belong in this submission.
FINAL = SELECTED

Xtr, Xte = build_features(train, test, FINAL)
model = make_models()["RF"].fit(Xtr, y)
pred = model.predict(Xte).astype(bool)

sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Transported": pred})
sub.to_csv("../submissions/sub02_rf_selected.csv", index=False)
print(sub.shape, sub["Transported"].mean().round(3))   # expect (4277, 2) and a mean near 0.5

(4277, 2) 0.478


In [29]:
sub.head

<bound method NDFrame.head of      PassengerId  Transported
0        0013_01         True
1        0018_01        False
2        0019_01         True
3        0021_01         True
4        0023_01         True
...          ...          ...
4272     9266_02         True
4273     9269_01        False
4274     9271_01         True
4275     9273_01         True
4276     9277_01        False

[4277 rows x 2 columns]>

In [30]:
lg = pd.read_csv(LOG)
lg.loc[lg["change"].str.startswith("SELECTED = rules+groupfill+cabin [RF]"), "public_lb"] = 0.79494
lg.to_csv(LOG, index=False)
lg.to_csv("../reports/experiment_log.csv", index=False)

Step 5: Compare boosted models, tune, blend, submit

In [31]:
#Cell 1: imports and a paired-score helper
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Why: cv_accuracy returned only mean/std. For a PAIRED comparison we need all 15 scores,
# in the same order for every model (same seeds, same splits). Fold difficulty then cancels
# out, so "model A minus model B" per fold is a much sharper test than comparing two means.
def cv_scores(model, X, n_repeats=3):
    out = []
    for r in range(n_repeats):
        splits = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(X, y, groups))
        out.extend(cross_val_score(model, X, y, cv=splits, scoring="accuracy", n_jobs=-1))
    return np.array(out)

FINAL = ("rules", "groupfill", "cabin")
Xtr, Xte = build_features(train, test, FINAL)

In [32]:
#Cell 2: compare all five models, untuned
# DECISION: one-hot encoding for every model, so the comparison differs only in the learner.
# Why: CatBoost can take categoricals natively, but that would change two things at once
# (model and encoding) and we couldn't say which one caused a difference.
# DECISION: shallow trees, small learning rate, subsampling. Why: ~8.7k rows overfit easily,
# as the Ames project showed. n_jobs/thread_count=1 because cross_val_score already
# parallelises across folds, and nesting parallelism slows things down.
def pre(): return make_preprocessor(False)    # trees need no scaling

untuned = dict(
    xgb=dict(n_estimators=400, learning_rate=0.03, max_depth=4, subsample=0.8,
             colsample_bytree=0.6, min_child_weight=3, reg_lambda=1.0),
    lgbm=dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=20,
              subsample=0.8, colsample_bytree=0.6),
)
def booster(kind, p):
    if kind == "xgb":
        return XGBClassifier(**p, eval_metric="logloss", random_state=SEED, n_jobs=1)
    return LGBMClassifier(**p, subsample_freq=1, random_state=SEED, n_jobs=1, verbose=-1)

candidates = {
    "Logistic": make_models()["Logistic"],
    "RF": make_models()["RF"],
    "XGBoost": Pipeline([("prep", pre()), ("m", booster("xgb", untuned["xgb"]))]),
    "LightGBM": Pipeline([("prep", pre()), ("m", booster("lgbm", untuned["lgbm"]))]),
    "CatBoost": Pipeline([("prep", pre()), ("m", CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, verbose=0,
        random_seed=SEED, thread_count=1))]),
}

scores = {}
for name, m in candidates.items():
    scores[name] = cv_scores(m, Xtr)
    s = scores[name]
    print(f"{name:10s} grouped CV {s.mean():.4f} ± {s.std():.4f}")
    log("5 models", f"{name} untuned on FINAL", s.mean(), s.std())

print("\nPaired difference vs RF (positive = better than RF):")
for name in ["Logistic", "XGBoost", "LightGBM", "CatBoost"]:
    d = scores[name] - scores["RF"]
    print(f"{name:10s} mean {d.mean():+.4f} | beats RF on {(d > 0).sum()}/15 folds")

Logistic   grouped CV 0.7904 ± 0.0113
RF         grouped CV 0.8088 ± 0.0085
XGBoost    grouped CV 0.8093 ± 0.0084
LightGBM   grouped CV 0.8096 ± 0.0080
CatBoost   grouped CV 0.8125 ± 0.0089

Paired difference vs RF (positive = better than RF):
Logistic   mean -0.0184 | beats RF on 0/15 folds
XGBoost    mean +0.0005 | beats RF on 8/15 folds
LightGBM   mean +0.0008 | beats RF on 9/15 folds
CatBoost   mean +0.0037 | beats RF on 11/15 folds


In [33]:
#Cell 3: tune the best booster with nested CV
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
KIND = "xgb"        # <- set from Cell 2

def space(trial, kind):
    # Why these ranges: shallow-to-moderate trees, slow learning, real regularisation.
    # Ames showed the surface is flat on ~1k rows, so we don't need wide ranges.
    p = dict(n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
             learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
             subsample=trial.suggest_float("subsample", 0.5, 1.0),
             colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0))
    if kind == "xgb":
        p.update(max_depth=trial.suggest_int("max_depth", 2, 6),
                 min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                 reg_lambda=trial.suggest_float("reg_lambda", 0.1, 30, log=True))
    else:
        p.update(num_leaves=trial.suggest_int("num_leaves", 4, 31),
                 min_child_samples=trial.suggest_int("min_child_samples", 5, 60),
                 reg_lambda=trial.suggest_float("reg_lambda", 0.1, 30, log=True))
    return p

def tune(kind, X, yv, gv, n_trials, seed=SEED):
    # Inner CV is GROUPED too: tuning on a leaky split would pick parameters that
    # memorise groups. 3 folds keeps each trial fast.
    inner = list(StratifiedGroupKFold(3, shuffle=True, random_state=seed + 1).split(X, yv, gv))
    def objective(trial):
        m = Pipeline([("prep", pre()), ("m", booster(kind, space(trial, kind)))])
        return cross_val_score(m, X, yv, cv=inner, scoring="accuracy", n_jobs=-1).mean()
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    st.optimize(objective, n_trials=n_trials)
    return st

# NESTED CV. Why: if Optuna picks parameters on the same folds we report, the score is
# optimistic. Here each outer fold is tuned only on its own training part, then scored on
# the untouched outer fold. The untuned model is scored on the SAME outer folds (paired).
outer = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(Xtr, y, groups))
tuned_s, untuned_s = [], []
for i, (tr, va) in enumerate(outer):
    Xa, ya, ga = Xtr.iloc[tr], y.iloc[tr], groups.iloc[tr]
    st = tune(KIND, Xa, ya, ga, n_trials=25)
    best = Pipeline([("prep", pre()), ("m", booster(KIND, st.best_params))]).fit(Xa, ya)
    base = Pipeline([("prep", pre()), ("m", booster(KIND, untuned[KIND]))]).fit(Xa, ya)
    tuned_s.append((best.predict(Xtr.iloc[va]) == y.iloc[va]).mean())
    untuned_s.append((base.predict(Xtr.iloc[va]) == y.iloc[va]).mean())
    print(f"Fold {i+1}: tuned {tuned_s[-1]:.4f} | untuned {untuned_s[-1]:.4f} | {st.best_params}")

d = np.array(tuned_s) - np.array(untuned_s)
print(f"\nNested tuned {np.mean(tuned_s):.4f} | untuned (same folds) {np.mean(untuned_s):.4f} "
      f"| gain {d.mean():+.4f}")
log("5 tuning", f"{KIND} tuned (nested)", np.mean(tuned_s), np.std(tuned_s))

Fold 1: tuned 0.8068 | untuned 0.8091 | {'n_estimators': 300, 'learning_rate': 0.09323621351781479, 'subsample': 0.8875664116805573, 'colsample_bytree': 0.9576492590949324, 'max_depth': 6, 'min_child_weight': 6, 'reg_lambda': 19.21295925538639}
Fold 2: tuned 0.8194 | untuned 0.8263 | {'n_estimators': 800, 'learning_rate': 0.039913058785616795, 'subsample': 0.8540362888980227, 'colsample_bytree': 0.3144091460070617, 'max_depth': 6, 'min_child_weight': 9, 'reg_lambda': 0.33572967053517927}
Fold 3: tuned 0.8097 | untuned 0.8120 | {'n_estimators': 600, 'learning_rate': 0.01637025907843186, 'subsample': 0.8475679561543505, 'colsample_bytree': 0.5827853146904393, 'max_depth': 6, 'min_child_weight': 4, 'reg_lambda': 4.602164908867431}
Fold 4: tuned 0.8067 | untuned 0.8078 | {'n_estimators': 700, 'learning_rate': 0.020165721691808594, 'subsample': 0.5488360570031919, 'colsample_bytree': 0.7789631185585097, 'max_depth': 4, 'min_child_weight': 2, 'reg_lambda': 1.6850517723339091}
Fold 5: tuned 0

In [34]:
#Cell 4: blend (out-of-fold, no weight search)
from sklearn.model_selection import cross_val_predict

# Why a blend: Ames gave a small gain from averaging two models. Here the members differ
# more (a linear model plus two tree models), so errors may be less correlated.
# DECISION: members use UNTUNED boosters, so the blend's CV stays honest (no tuning on these
# folds). Logistic gets the cabinbin block because it passed the rule for Logistic only;
# the trees use FINAL.
Xlog, _ = build_features(train, test, FINAL + ("cabinbin",))
members = {
    "Logistic": (make_models()["Logistic"], Xlog),
    "RF": (make_models()["RF"], Xtr),
    "XGBoost": (candidates["XGBoost"], Xtr),
}
repeat_splits = [list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r)
                      .split(Xtr, y, groups)) for r in range(3)]

def fold_acc(prob, splits):
    return np.array([((prob[va] > 0.5) == y.iloc[va].values).mean() for _, va in splits])

res = {k: [] for k in list(members) + ["Blend (equal avg)", "Blend RF+XGB"]}
for splits in repeat_splits:
    oof = {k: cross_val_predict(m, X, y, cv=splits, method="predict_proba", n_jobs=-1)[:, 1]
           for k, (m, X) in members.items()}
    for k in members:
        res[k].extend(fold_acc(oof[k], splits))
    res["Blend (equal avg)"].extend(fold_acc(np.mean(list(oof.values()), axis=0), splits))
    res["Blend RF+XGB"].extend(fold_acc((oof["RF"] + oof["XGBoost"]) / 2, splits))

for k, v in res.items():
    v = np.array(v); print(f"{k:18s} {v.mean():.4f} ± {v.std():.4f}")
    log("5 blend", k, v.mean(), v.std())

Logistic           0.7950 ± 0.0099
RF                 0.8088 ± 0.0085
XGBoost            0.8093 ± 0.0084
Blend (equal avg)  0.8104 ± 0.0092
Blend RF+XGB       0.8121 ± 0.0079


In [36]:
#Cell 5: choose the final model by the pre-stated rules, then submit
# Fill these in from Cells 2-4 BEFORE running (rule 1: needs >= +0.002 over RF's 0.8088).
CHOICE = "CatBoost"    # one of: "RF", "XGBoost", "LightGBM", "CatBoost", "Blend"
# Params to use if CHOICE is a booster AND nested tuning passed rule 2; else leave None.
TUNED_PARAMS = None

if CHOICE == "Blend":
    ps = [make_models()["Logistic"].fit(Xlog, y).predict_proba(build_features(train, test, FINAL + ("cabinbin",))[1])[:, 1],
          make_models()["RF"].fit(Xtr, y).predict_proba(Xte)[:, 1],
          candidates["XGBoost"].fit(Xtr, y).predict_proba(Xte)[:, 1]]
    pred = (np.mean(ps, axis=0) > 0.5)
else:
    m = candidates[CHOICE]
    if TUNED_PARAMS and CHOICE in ("XGBoost", "LightGBM"):
        m = Pipeline([("prep", pre()), ("m", booster("xgb" if CHOICE == "XGBoost" else "lgbm", TUNED_PARAMS))])
    pred = m.fit(Xtr, y).predict(Xte).astype(bool)

sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Transported": pred})
sub.to_csv("../submissions/sub04_catboost.csv", index=False)   # new name, don't overwrite sub03
print(sub.shape, sub["Transported"].mean().round(3))   # should differ from 0.478

(4277, 2) 0.51
